In [1]:
import pandas as pd
import numpy as np

print("Iniciando a leitura e cruzamento dos dados...")

# 1. Leitura dos Dados Brutos
caminho = '../data/raw/'
df_orders = pd.read_csv(f'{caminho}olist_orders_dataset.csv')
df_items = pd.read_csv(f'{caminho}olist_order_items_dataset.csv')
df_customers = pd.read_csv(f'{caminho}olist_customers_dataset.csv')
df_sellers = pd.read_csv(f'{caminho}olist_sellers_dataset.csv')
df_products = pd.read_csv(f'{caminho}olist_products_dataset.csv')

Iniciando a leitura e cruzamento dos dados...


In [2]:
# 2. Cruzamentos (JOINs)
# Começamos com os pedidos e vamos adicionando as camadas de informação
df = pd.merge(df_orders, df_customers[['customer_id', 'customer_state']], on='customer_id', how='inner')
df = pd.merge(df, df_items[['order_id', 'product_id', 'seller_id', 'price', 'freight_value']], on='order_id', how='inner')
df = pd.merge(df, df_sellers[['seller_id', 'seller_state']], on='seller_id', how='inner')
df = pd.merge(df, df_products[['product_id', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']], on='product_id', how='inner')

# Filtrando apenas os pedidos que já foram entregues para treinar o modelo
df = df[df['order_status'] == 'delivered'].copy()

In [3]:
# 3. Engenharia de Features (Criando as variáveis de ouro para o modelo)

# Transformando colunas de texto em data
colunas_data = ['order_purchase_timestamp', 'order_delivered_customer_date', 'order_estimated_delivery_date']
for col in colunas_data:
    df[col] = pd.to_datetime(df[col])

# A) Variável Alvo: Atrasou? (1 = Sim, 0 = Não)
df['dias_atraso'] = (df['order_delivered_customer_date'] - df['order_estimated_delivery_date']).dt.days
df['houve_atraso'] = df['dias_atraso'].apply(lambda x: 1 if x > 0 else 0)

# B) Variável Logística: Rota (Origem -> Destino)
df['rota_logistica'] = df['seller_state'] + "->" + df['customer_state']

# C) Variável Física: Volume Cúbico (em cm³) e Valor Total
df['volume_cm3'] = df['product_length_cm'] * df['product_height_cm'] * df['product_width_cm']
df['valor_total_pedido'] = df['price'] + df['freight_value']

# D) Variável Temporal: Mês e Dia da Semana da Compra
df['mes_compra'] = df['order_purchase_timestamp'].dt.month
df['dia_semana_compra'] = df['order_purchase_timestamp'].dt.dayofweek

In [4]:
# 4. Seleção Final e Limpeza
colunas_finais = [
    'order_id', 'order_purchase_timestamp', 'rota_logistica', 'seller_state', 'customer_state',
    'price', 'freight_value', 'valor_total_pedido', 'product_weight_g', 'volume_cm3', 
    'mes_compra', 'dia_semana_compra', 'houve_atraso'
]

df_final = df[colunas_finais].dropna().copy()

In [5]:
# 5. Exportação
caminho_saida = '../data/processed/base_logistica_enriquecida.csv'
df_final.to_csv(caminho_saida, index=False)

print(f"Base consolidada gerada com sucesso! Total de registros: {df_final.shape[0]}")
print(f"Arquivo salvo em: {caminho_saida}")

Base consolidada gerada com sucesso! Total de registros: 110179
Arquivo salvo em: ../data/processed/base_logistica_enriquecida.csv
